In [1]:
import pandas as pd
import spacy
from IPython.core.magic import register_cell_magic
from google_play_scraper import reviews_all
from sklearn.feature_extraction.text import CountVectorizer
import seaborn as sns
import matplotlib.pyplot as plt
import time

In [ ]:
@register_cell_magic
def skip(line, cell):
    return

przygotowanie danych

In [2]:
%%skip
languages = ["pl"]
apps_ids = ["pl.mapa_turystyczna.app", "de.komoot.android", "com.strava", "com.alltrails.alltrails", "com.meteoblue.droid", "menion.android.locus", "cz.seznam.mapy", "com.pagasolutions.emergencycall_pl", "info.burzowo", "com.windyty.android", "app.com.example.szymi.myapplication", "pl.traseo2", "no.nrk.yr", "pl.janpogocki.icm.pogoda", "com.verdant.kgp", "com.verdant.wkt", "org.peakfinder.area.alps", "com.google.android.apps.maps"]

parsed_list = []

for count, app_id in enumerate(apps_ids, start=1):
    print(f"{app_id} is running. {count}/{len(apps_ids)}")
    time.sleep(7)
    for lang in languages:
        parsed_dict = reviews_all(app_id=app_id, sleep_milliseconds=1200, lang=lang)
        partial_df = pd.DataFrame(parsed_dict).assign(appId=app_id, language=lang)
        partial_df = partial_df.drop(columns=["userImage", "userName"])
        parsed_list.append(partial_df)

df = pd.concat(parsed_list, ignore_index=True)
df.to_csv("reviews_data.csv", index=False)
print("success")

pl.mapa_turystyczna.app is running. 1/18
de.komoot.android is running. 2/18
com.strava is running. 3/18
com.alltrails.alltrails is running. 4/18
com.meteoblue.droid is running. 5/18
menion.android.locus is running. 6/18
cz.seznam.mapy is running. 7/18
com.pagasolutions.emergencycall_pl is running. 8/18
info.burzowo is running. 9/18
com.windyty.android is running. 10/18
app.com.example.szymi.myapplication is running. 11/18
pl.traseo2 is running. 12/18
no.nrk.yr is running. 13/18
pl.janpogocki.icm.pogoda is running. 14/18
com.verdant.kgp is running. 15/18
com.verdant.wkt is running. 16/18
org.peakfinder.area.alps is running. 17/18
com.google.android.apps.maps is running. 18/18
success


In [4]:
df.columns
df.head(1)

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,appId,language
0,f3da5f79-0d6c-490b-b8e3-20821ca42e2f,ok,5,0,1.15.11,2026-04-17 18:46:24,NaN,NaN,1.15.11,pl.mapa_turystyczna.app,pl


In [7]:
df = pd.read_csv(
    "reviews_data.csv",
    dtype={
        "appId": str,
        "language": str,
        "reviewId": str,
        "score": "int8",
        "content": str,
        "thumbsUpCount": "int32",
        "reviewCreatedVersion": str,
        "replyContent": str,
        "appVersion": str,
    },
    parse_dates=["at", "repliedAt"],
)
app_ids = df["appId"].unique()
languages = df["language"].unique()

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 157580 entries, 0 to 157579
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   reviewId              157580 non-null  str           
 1   content               157411 non-null  str           
 2   score                 157580 non-null  int8          
 3   thumbsUpCount         157580 non-null  int32         
 4   reviewCreatedVersion  153056 non-null  str           
 5   at                    157580 non-null  datetime64[us]
 6   replyContent          16143 non-null   str           
 7   repliedAt             16143 non-null   datetime64[us]
 8   appVersion            153056 non-null  str           
 9   appId                 157580 non-null  str           
 10  language              157580 non-null  str           
dtypes: datetime64[us](2), int32(1), int8(1), str(7)
memory usage: 11.6 MB


przetwarzanie danych językowych

In [22]:
nlp = spacy.load("pl_core_news_lg")


def lemmatize(text):
    if pd.isna(text):
        return ""
    doc = nlp(str(text).lower())

    lemmas = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop]

    return " ".join(lemmas)

In [23]:
def get_ngram_counts(
    df: pd.DataFrame,
    text_col: str = "lemmas",
    ngram_range: tuple = (2, 3),
    min_size: int = 5,
):
    print("starting conversion")

    vectorizer = CountVectorizer(ngram_range=ngram_range, min_df=min_size)
    X = vectorizer.fit_transform(df[text_col])

    ngram_counts = pd.DataFrame(
        X.sum(axis=0).T, index=vectorizer.get_feature_names_out(), columns=["count"]
    )
    return ngram_counts


def backwards_ngram(df, phrase, text_col: str = "lemmas"):
    results = df[df[text_col].str.contains(phrase, na=False)]
    print(f"found {len(results)} reviews for the phrase '{phrase}'")
    return results

In [24]:
def review_prep(df: pd.DataFrame, score_range: tuple):
    result = df[
        (df["score"] >= score_range[0]) & (df["score"] <= score_range[1])
    ].assign(lemmas=lambda x: x["content"].apply(lemmatize))
    return result

In [25]:
reviews_ready = review_prep(df, score_range=(0, 2))
done_apps = []
for app_id in apps_ids:
    app_ngram = (
        get_ngram_counts(reviews_ready[reviews_ready["appId"] == app_id])
        .assign(appId=app_id)
        .reset_index(names="keywords")
    )
    done_apps.append(app_ngram)
all_apps = pd.concat(done_apps, ignore_index=True)

NameError: name 'apps_ids' is not defined

In [ ]:
%%skip
ngram_short = ngram.head(10)

plt.figure(figsize=(12,6))
ax = (sns.barplot(data=ngram_short, x=ngram_short.index, y="count"))

ax.set_xticks(range(len(ngram_short)))
ax.set_xticklabels(rotation=-30, labels=ngram_short.index)
ax.set_title("Najpopularniejsze zbitki słów w negatywnych recenzjach Mapy Turystycznej")
ax.set_ylabel("Licznik")
ax.set_xlabel("Zbitki słów")
plt.show()